Version 3a New features:
*   Full xml articlde download using e-utilities, AWS or FTP
*   Better tracking of download summary and missing PMCID


In [1]:
import requests
import xml.etree.ElementTree as ET
import time
import os
import logging
import gzip
import shutil
import pandas as pd
import json
import tarfile
from datetime import datetime, timedelta
from google.colab import drive, userdata
from urllib.parse import urlencode

In [2]:
# Variable declarations at start
BASE_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
RATE_LIMIT_DELAY = 0.34
DATABASE = "pubmed"
RETMODE_XML = "xml"
USE_HISTORY = "y"
DRIVE_BASE = "/content/drive/MyDrive/AI6129"
XML_PATH = f"{DRIVE_BASE}/xml"
LOG_PATH = f"{DRIVE_BASE}/logs"
TRACKING_FILE = f"{DRIVE_BASE}/download_tracker.csv"
MISSING_PMCID_LOG = f"{DRIVE_BASE}/missing_pmcids"
AWS_XML_PATH = f"{DRIVE_BASE}/xml/AWS"
FTP_XML_PATH = f"{DRIVE_BASE}/xml/FTP"
FTP_RATE_LIMIT_DELAY = 0.5
NCBI_API_KEY = userdata.get('ENTREZ_API')


In [3]:
# Configurations
PATHOGENS = {
    "hepatitis_a": {
        "name": "Hepatitis A",
        "mesh": "Hepatitis A",
        "term": "Hepatitis A"  # Single term, no synonyms
    },
    "hepatitis_e": {
        "name": "Hepatitis E",
        "mesh": "Hepatitis E",
        "term": "Hepatitis E"  # Single term, no synonyms
    }
}

# PMC specific URLs
PMC_OA_SERVICE = "https://www.ncbi.nlm.nih.gov/pmc/utils/oa/oa.fcgi"
PMC_PDF_BASE = "https://www.ncbi.nlm.nih.gov/pmc/articles"

# Enhanced rate limiting
PMC_RATE_LIMIT_DELAY = 1.0  # PMC requires slower rate
MAX_RETRIES = 3
BATCH_SIZE = 10

# File size limits (in MB)
MAX_XML_SIZE_MB = 10
COMPRESS_THRESHOLD_MB = 1

# Logging configuration
LOG_LEVEL = logging.INFO
LOG_FORMAT = '%(asctime)s - %(levelname)s - %(funcName)s - %(message)s'

In [4]:
def calculate_pmcid_prefix(pmcid_numeric):
    pmcid_clean = str(pmcid_numeric).replace('PMC', '').replace('pmc', '')
    pmcid_padded = pmcid_clean.zfill(4)
    first_four = pmcid_padded[:4]
    prefix1 = first_four[:2]
    prefix2 = first_four[2:4]
    prefix = f"{prefix1}/{prefix2}"
    return prefix

def create_method_directories():
    os.makedirs(AWS_XML_PATH, exist_ok=True)
    os.makedirs(FTP_XML_PATH, exist_ok=True)
    logging.info(f"Created AWS directory: {AWS_XML_PATH}")
    logging.info(f"Created FTP directory: {FTP_XML_PATH}")

In [5]:
#AWS S3 Download Functions
def construct_aws_https_url(pmcid):
    pmcid_clean = pmcid.replace('PMC', '') if pmcid.startswith('PMC') else pmcid
    prefix = calculate_pmcid_prefix(pmcid_clean)
    base_url = "https://ftp.ncbi.nlm.nih.gov/pub/pmc/oa_package"
    url = f"{base_url}/{prefix}/PMC{pmcid_clean}.xml"
    return url

def download_from_aws_s3(pmcid, search_date, retry_count=0):
    try:
        pmcid_clean = pmcid.replace('PMC', '') if pmcid.startswith('PMC') else pmcid
        url = construct_aws_https_url(pmcid)

        logging.info(f"Attempting AWS download for PMC{pmcid_clean}")
        logging.debug(f"AWS URL: {url}")

        time.sleep(FTP_RATE_LIMIT_DELAY)

        response = requests.get(url, timeout=60)

        if response.status_code == 200:
            xml_content = response.text

            if '<article' in xml_content or '<pmc-articleset>' in xml_content:
                filepath = os.path.join(AWS_XML_PATH, f"PMC{pmcid_clean}_{search_date}.xml")

                if os.path.exists(filepath) or os.path.exists(f"{filepath}.gz"):
                    logging.info(f"AWS XML for PMC{pmcid_clean} already exists - skipping")
                    return True, url

                with open(filepath, 'w', encoding='utf-8') as f:
                    f.write(xml_content)

                file_size_mb = os.path.getsize(filepath) / (1024 * 1024)
                logging.info(f"Saved AWS XML for PMC{pmcid_clean}: {file_size_mb:.2f} MB")

                if file_size_mb > COMPRESS_THRESHOLD_MB:
                    compressed_filepath = f"{filepath}.gz"
                    with open(filepath, 'rb') as f_in:
                        with gzip.open(compressed_filepath, 'wb') as f_out:
                            shutil.copyfileobj(f_in, f_out)
                    os.remove(filepath)
                    compressed_size_mb = os.path.getsize(compressed_filepath) / (1024 * 1024)
                    logging.info(f"Compressed AWS XML for PMC{pmcid_clean}: {compressed_size_mb:.2f} MB")

                return True, url
            else:
                logging.warning(f"Invalid XML content from AWS for PMC{pmcid_clean}")
                return False, url

        elif response.status_code == 404:
            logging.warning(f"Article not found in AWS S3 for PMC{pmcid_clean}")
            return False, url

        else:
            raise Exception(f"HTTP {response.status_code}")

    except Exception as e:
        logging.error(f"AWS download failed for PMC{pmcid_clean}: {str(e)}")

        if retry_count < MAX_RETRIES:
            logging.info(f"Retrying AWS download for PMC{pmcid_clean} (attempt {retry_count + 1})")
            time.sleep(RATE_LIMIT_DELAY * 2)
            return download_from_aws_s3(pmcid, search_date, retry_count + 1)

        return False, url if 'url' in locals() else None

In [6]:
#FTP package download functions

def construct_ftp_package_url(pmcid):
    pmcid_clean = pmcid.replace('PMC', '') if pmcid.startswith('PMC') else pmcid
    prefix = calculate_pmcid_prefix(pmcid_clean)
    base_url = "https://ftp.ncbi.nlm.nih.gov/pub/pmc/oa_package"
    url = f"{base_url}/{prefix}/PMC{pmcid_clean}.xml.zip"
    return url

def extract_xml_from_tar_gz(tar_path, pmcid):
  try:
    pmcid_clean = pmcid.replace('PMC', '') if pmcid.startswith('PMC') else pmcid

    with tarfile.open(tar_path, 'r:gz') as tar:
      members = tar.getmembers()

      xml_extensions = ['.xml', '.XML', '.nxml']
      xml_member = None

      for member in members:
        for ext in xml_extensions:
          if member.name.endswith(ext) and not member.name.startswith('.'):
              xml_members = [member]
              break
          if xml_member:
            break

      if xml_member:
        xml_file = tar.extractfile(xml_member)
        if xml_file:
          xml_content = xml_file.read().decode('utf-8')
          logging.info(f"Extracted {xml_member.name} from FTP package for PMC{pmcid_clean}")
          return xml_content
        else:
          logging.error(f"Could not extract file {xml_member.name}")
          return None
      else:
        logging.error(f"No XML file found in FTP package for PMC{pmcid_clean}")
        logging.debug(f"Package contents: {[m.name for m in members]}")
        return None

  except Exception as e:
    logging.error(f"Failed to extract XML from tar.gz for PMC{pmcid}: {str(e)}")

    return None


def download_from_ftp_package(pmcid, search_date, retry_count=0):

  try:
    pmcid_clean = pmcid.replace('PMC', '') if pmcid.startswith('PMC') else pmcid
    package_url = construct_ftp_package_url(pmcid)

    logging.info(f"Downloading FTP package for PMC{pmcid_clean}")
    logging.debug(f"FTP URL: {package_url}")

    time.sleep(FTP_RATE_LIMIT_DELAY)

    response = requests.get(package_url, timeout=120, stream=True)

    if response.status_code == 200:
      temp_package = f"/tmp/PMC{pmcid_clean}.tar.gz"

      with open(temp_package, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
          if chunk:
            f.write(chunk)
      package_size_mb = os.path.getsize(temp_package) / (1024 * 1024)
      logging.info(f"Downloaded FTP package for PMC{pmcid_clean}: {package_size_mb:.2f} MB")

      xml_content = extract_xml_from_tar_gz(temp_package, pmcid)

      if xml_content:
        filepath = os.path.join(FTP_XML_PATH, f"PMC{pmcid_clean}_{search_date}.xml")

        if os.path.exists(filepath) or os.path.exists(f"{filepath}.gz"):
          logging.info(f"FTP XML for PMC{pmcid_clean} already exists - skiping")
          os.remove(temp_package)
          return True, package_url

        with open(filepath, 'w', encoding='utf-8') as f:
          f.write(xml_content)

        file_size_mb = os.path.getsize(filepath) / (1024 * 1024)
        logging.info(f"Saved FTP XML for PMC{pmcid_clean}: {file_size_mb:.2f} MB")

        if file_size_mb > COMPRESS_THRESHOLD_MB:
          compressed_filepath = f"{filepath}.gz"
          with open(filepath, 'rb') as f_in:
            with gzip.open(compressed_filepath, 'wb') as f_out:
              shutil.copyfileobj(f_in, f_out)
          os.remove(filepath)
          compressed_size_mb = os.path.getsize(compressed_filepath) / (1024 * 1024)
          logging.info(f"Compressed FTP XML for PMC{pmcid_clean}: {compressed_size_mb:.2f} MB")

        os.remove(temp_package)
        return True, package_url
      else:
        logging.warning(f"Failed to extract XML from FTP package for PMC{pmcid_clean}")
        if os.path.exists(temp_package):
          os.remove(temp_package)
        return False, package_url

    elif response.status_code == 404:
      logging.warning(f"FTP package not found for PMC{pmcid_clean}")
      return False, package_url

    else:
      raise Exception(f"HTTP {response.status_code}")

  except Exception as e:
    logging.error(f"Failed to construct FTP package URL for PMC{pmcid}: {str(e)}")

  if retry_count < MAX_RETRIES:
    logging.info(f"Retrying FTP download for PMC{pmcid} (attempt {retry_count + 1})")
    time.sleep(RATE_LIMIT_DELAY * 2)
    return download_from_ftp_package(pmcid, search_date, retry_count + 1)

  if 'package_url' in locals():
    return False, package_url
  else:
    return None



In [7]:
#Logging Functions
def setup_logging():
  os.makedirs(LOG_PATH, exist_ok=True)
  os.makedirs(MISSING_PMCID_LOG, exist_ok=True)

  log_filename = f"{LOG_PATH}/pubmed_download_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

  logging.basicConfig(
      level=LOG_LEVEL,
      format=LOG_FORMAT,
      handlers=[logging.FileHandler(log_filename, encoding = 'utf-8'),
                logging.StreamHandler()],
      force=True,
  )

  logging.info(f"Logging initialized. Log file: {log_filename}")
  logging.info("PubMed Article Downloader v3")


def mount_google_drive():
  try:
    drive.mount('/content/drive', force_remount=False)
    os.makedirs(XML_PATH, exist_ok=True)
    os.makedirs(LOG_PATH, exist_ok=True)
    os.makedirs(MISSING_PMCID_LOG, exist_ok=True)
    os.makedirs(AWS_XML_PATH, exist_ok=True)  # NEW
    os.makedirs(FTP_XML_PATH, exist_ok=True)
    logging.info("Google Drive mounted successfully")
    return True
  except Exception as e:
    logging.error(f"Failed to mount Google Drive: {str(e)}")
    return False

In [8]:
def build_pathogen_search_query(pathogen_key):
  """
    Build PubMed search query for single pathogen

    Args:
      pathogen_key: Key for pathogen in PATHOGENS dict

    Returns:
      str: Formatted PubMed search query
  """

  pathogen = PATHOGENS[pathogen_key]
  term = pathogen['term']
  mesh = pathogen['mesh']

  full_query = (f'(({term}[Title] OR {term}[Abstract] OR "{term}"[Methods - Key Terms]) '
                f'OR ({mesh}[MeSH Terms] OR {term}[All Fields]) '
                f'NOT ((review[All Fields] OR "review literature as topic"[MeSH Terms] OR review[All Fields]) '
                f'NOT Overview[All Fields]))')

  logging.info(f"Built search query for {pathogen['name']}")
  logging.info(f"Query: {full_query}")

  return full_query

In [9]:
#XML manipulation functions

def fetch_pmc_fulltext_xml(pmcid, retry_count=0):
  try:
    url = construct_pmc_fetch_url(pmcid)
    pmcid_clean = pmcid.replace('PMC', '', 1) if pmcid.startswith('PMC') else pmcid
    logging.info(f"Fetching full text XML for {pmcid_clean}")

    time.sleep(PMC_RATE_LIMIT_DELAY)
    response = requests.get(url, timeout=30)

    if response.status_code == 200:
      xml_content = response.text

      if '<body>' in xml_content or '<abstract>' in xml_content:
        logging.info(f"Successfully retrieved full-text XML for PMC{pmcid_clean}")
        return xml_content
      else:
        logging.warning(f"Response may contain metadata only for PMC{pmcid_clean}")
        return xml_content
    elif response.status_code == 404:
      logging.error(f"Full-text XML not found for PMC{pmcid_clean}")
      return None
    else:
      raise Exception(f"HTTP {response.status_code}")

  except Exception as e:
    logging.error(f"Failed to fetch full-text for PMC{pmcid_clean}: {str(e)}")

    if retry_count < MAX_RETRIES:
      logging.info(f". Retrying. Fetching full-text for PMC{pmcid_clean} (attempt {retry_count + 1})")
      time.sleep(RATE_LIMIT_DELAY * 2)
      return fetch_pmc_fulltext_xml(pmcid, retry_count + 1)

    return None

def save_xml_with_compression(xml_content, identifier, search_date):
  try:
    filename = f"{identifier}_{search_date}.xml"
    filepath = os.path.join(XML_PATH, filename)

    #Check if already exists
    if os.path.exists(filepath) or  os.path.exists(f"{filepath}.gz"):
      logging.info(f"XML for {identifier} already exists - skipping")
      return True

    #write XML
    with open(filepath, 'w', encoding='utf-8') as f:
      f.write(xml_content)

    #check file size
    file_size_mb = os.path.getsize(filepath) / (1024 * 1024)
    logging.info(f"Saved XML for {identifier}: {file_size_mb:.2f} MB")

    #compress if over threshold
    if file_size_mb > COMPRESS_THRESHOLD_MB:
      compressed_filepath = f"{filepath}.gz"
      with open(filepath, 'rb') as f_in:
        with gzip.open(compressed_filepath, 'wb') as f_out:
          shutil.copyfileobj(f_in, f_out)

      #remove original if compression successful
      os.remove(filepath)
      compressed_size_mb = os.path.getsize(compressed_filepath) / (1024 * 1024)
      logging.info(f"Compressed XML for {identifier}: {compressed_size_mb:.2f} MB")

    return True

  except Exception as e:
    logging.error(f"Failed to save XML for  {identifier}: {str(e)}")
    return False


In [10]:
#Extract functions
def extract_pmid_list(xml_response):
    root = ET.fromstring(xml_response)
    pmid_list = []
    id_list = root.find('IdList')

    if id_list is not None:
        for id_element in id_list.findall('Id'):
          pmid_list.append(id_element.text)
    return pmid_list

def extract_publication_date(article):

  pub_date = None
  data_element = article.find('.//PubDate')
  if data_element is not None:
    year = data_element.find('.//Year')
    month = data_element.find('.//Month')
    day = data_element.find('.//Day')

    if year is not None:
      pub_date = f"{year.text}"
      if month is not None:
        pub_date += f"-{month.text}"
        if day is not None:
          pub_date += f"-{day.text}"
  return pub_date

def extract_abstract_text(article):
  abstract = "Abstract not available"
  abstract_element = article.find('.//Abstract')
  if abstract_element is not None:
    abstract_texts = []
    for abstract_text in abstract_element.findall('AbstractText'):
      label = abstract_text.get('Label', '')
      text = abstract_text.text if abstract_text.text is not None else ''

      if label and text:
        abstract_texts.append(f'{label}: {text}')
      elif text:
        abstract_texts.append(text)

    if abstract_texts:
      abstract = ' '.join(abstract_texts)

  return abstract

def extract_total_count(xml_response):
  root = ET.fromstring(xml_response)
  count_element = root.find('Count')

  if count_element is not None and count_element.text is not None:
    return int(count_element.text)
  return 0

In [11]:
def convert_pmid_to_pmcid(pmid, retry_count=0):
  try:
    url = construct_elink_url(pmid)

    logging.debug(f"Converting PMID {pmid} to PMCID")
    time.sleep(RATE_LIMIT_DELAY)

    response = requests.get(url, timeout=30)
    if response.status_code != 200:
      logging.warning(f"elink returned HTTP {response.status_code} for PMID {pmid}")
      return None

    root = ET.fromstring(response.text)
    linksetdb = root.find('.//LinkSetDb')

    if linksetdb is None:
      logging.warning(f"No PMC link found for PMID {pmid}")
      return None

    link_id = linksetdb.find('.//Link/Id')
    if link_id is not None and link_id.text:
      pmcid_num = link_id.text
      pmcid = 'PMC' + pmcid_num
      logging.info(f"Converted PMID {pmid} to PMCID {pmcid}")
      return pmcid

    logging.warning(f"No PMCID found in response for PMID {pmid}")
    return None

  except ET.ParseError as e:
    logging.error(f"Failed to parse XML for PMID {pmid}: {str(e)}")
    return None

  except Exception as e:
    logging.error(f"Failed to convert PMID {pmid} to PMCID: {str(e)}")

    if retry_count < MAX_RETRIES:
      logging.info(f". Retrying. Converting PMID {pmid} (attempt {retry_count + 1})")
      time.sleep(RATE_LIMIT_DELAY * 2)
      return convert_pmid_to_pmcid(pmid, retry_count + 1)
  return None

def batch_convert_pmids_to_pmcids(pmid_list):
  pmid_to_pmcid_map = {}
  missing_pmcid_list = []

  logging.info(f"Starting PMID-to-PMCID conversion for {len(pmid_list)} articles")

  total_pmids = len(pmid_list)

  for i, pmid in enumerate(pmid_list, 1):
    pmcid = convert_pmid_to_pmcid(pmid)

    if pmcid is not None:
      pmid_to_pmcid_map[pmid] = pmcid
    else:
      missing_pmcid_list.append(pmid)
      logging.warning(f"PMID {pmid} has no PMC version")

  success_count = len(pmid_to_pmcid_map)
  success_rate = (success_count / total_pmids * 100) if total_pmids > 0 else 0

  logging.info(f"PMID-to-PMCID conversion completed. {success_count}/{total_pmids} successful ({success_rate:.1f}%)")
  logging.warning(f"Missing PMCIDs: {len(missing_pmcid_list)} articles")

  return pmid_to_pmcid_map, missing_pmcid_list

def track_missing_pmcid(tracking_df, pmid, reason):
  tracking_df = update_tracking_data(
      df=tracking_df,
      pmid=pmid,
      pmcid=None,
      pmcid_status='not_found',
      fulltext_xml_status='not_attempted',
      error_message=reason
  )

  missing_log_file = f"{MISSING_PMCID_LOG}/missing_pmcids_{datetime.now().strftime('%Y%m%d')}.csv"

  missing_record = {
      'pmid': pmid,
      'reason': reason,
      'date_checked': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
  }

  if os.path.exists(missing_log_file):
    missing_df = pd.read_csv(missing_log_file)
    missing_df = pd.concat([missing_df, pd.DataFrame([missing_record])], ignore_index=True)
  else:
    missing_df = pd.DataFrame([missing_record])

  missing_df.to_csv(missing_log_file, index=False)
  logging.warning(f"Tracked missing PMCID: PMID {pmid} - {reason}")

  return tracking_df

def generate_download_summary(tracking_df, date_start, date_end):
  summary_file = f"{LOG_PATH}/summary_{datetime.now().strftime('&Y%m%d')}.txt"

  current_date_str = datetime.now().strftime('%Y-%m-%d')
  current_run = tracking_df[tracking_df['download_date'].str.contains(current_date_str, na=False)]

  total_processed = len(current_run)
  pmcids_found = len(current_run[current_run['pmcid_status'] == 'found'])
  pmcids_not_found = len(current_run[current_run['pmcid_status'] == 'not_found'])
  #pmcids_fetch_failed = len(current_run[current_run['pmcid_status'] == 'fetch_failed'])

  fulltext_success = len(current_run[current_run['fulltext_xml_status'] == 'success'])
  fulltext_failed = len(current_run[current_run['fulltext_xml_status'] == 'failed'])
  #fulltext_not_attempted = len(current_run[current_run['fulltext_xml_status'] == 'not_attempted'])

  method_stats = {}
  for method in ['eutilities', 'aws', 'ftp']:
    method_df = current_run[current_run['download_method'] == method]
    method_stats[method] = {
        'total': len(method_df),
        'success': len(method_df[method_df['fulltext_xml_status'] == 'success']),
        'failed': len(method_df[method_df['fulltext_xml_status'] == 'failed']),
    }

  conversion_rate = (pmcids_found / total_processed * 100) if total_processed > 0 else 0
  success_rate = (fulltext_success / total_processed * 100) if total_processed > 0 else 0


  summary = f"""
    ===== PUBMED ARTICLE DOWNLOADER v3 - SUMMARY REPORT =====
    Date Range: {date_start} to {date_end}
    Execution Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

    PMID-TO-PMCID CONVERSION:
    -------------------------
    Total PMIDs Searched: {total_processed}
    PMCIDs Found: {pmcids_found} ({conversion_rate:.1f}%)
    PMCIDs Not Found: {pmcids_not_found}

    FULL-TEXT XML RETRIEVAL:
    ------------------------
    Successful Downloads: {fulltext_success} ({success_rate:.1f}% of found)
    Failed Downloads: {fulltext_failed}

    DOWNLOAD METHOD BREAKDOWN:
    --------------------------
    E-utilities Method:
    Total Attempts: {method_stats['eutilities']['total']}
    Successful: {method_stats['eutilities']['success']}
    Failed: {method_stats['eutilities']['failed']}

    AWS S3 Method:
    Total Attempts: {method_stats['aws']['total']}
    Successful: {method_stats['aws']['success']}
    Failed: {method_stats['aws']['failed']}

    FTP Package Method:
    Total Attempts: {method_stats['ftp']['total']}
    Successful: {method_stats['ftp']['success']}
    Failed: {method_stats['ftp']['failed']}

    HISTORICAL DATA:
    ----------------
    Total Records in Database: {len(tracking_df)}
    Total Unique PMCIDs: {len(tracking_df[tracking_df['pmcid'].notna()]['pmcid'].unique())}

    FILES GENERATED:
    ----------------
    Tracking File: {TRACKING_FILE}
    Missing PMCIDs Log: {MISSING_PMCID_LOG}/missing_pmcids_{datetime.now().strftime('%Y%m%d')}.csv
    XML Files Directory: {XML_PATH}
    AWS XML: {AWS_XML_PATH}
    FTP XML: {FTP_XML_PATH}
    =========================================================
    """

  with open(summary_file, 'w') as f:
    f.write(summary)

  logging.info(f"Summary report saved to {summary_file}")
  print(summary)

  if pmcids_not_found > 0:
    missing_pmids = current_run[current_run['pmcid_status'] == 'not_found']['pmid'].tolist()
    missing_summary_file = f"{LOG_PATH}/missing_pmcids_summary_{datetime.now().strftime('%Y%m%d')}.txt"

    with open(missing_summary_file, 'w') as f:
      f.write(f"Articles Without PMC Versions ({len(missing_pmids)} total):\n")
      f.write("="*60 + "\n\n")
      for pmid in missing_pmids:
          f.write(f"PMID: {pmid}\n")
      f.write("\n" + "="*60 + "\n")
      f.write("Note: These articles are not available in PubMed Central.\n")
      f.write("They may be closed-access or not submitted to PMC.\n")

    logging.info(f"Missing PMCIDs summary saved to {missing_summary_file}")

In [16]:
def construct_elink_url(pmid):
  params ={
        'dbfrom': 'pubmed',
        'db': 'pmc',
        'linkname': 'pubmed_pmc',
        'id': pmid,
        'retmode': 'xml',
        'api_key': NCBI_API_KEY
  }
  return BASE_URL + 'elink.fcgi?' + urlencode(params)

def construct_pmc_fetch_url(pmcid):
  pmcid_clean = pmcid.replace('PMC','') if pmcid.startswith('PMC') else pmcid
  params ={
        'db': 'pmc',
        'id': pmcid_clean,
        'retmode': 'xml',
        'api_key': NCBI_API_KEY
  }
  return BASE_URL + 'efetch.fcgi?' + urlencode(params)

def construct_esearch_url(database, term, mindate, maxdate, datetype, retmax, usehistory):
    params = {
        'db': database,
        'term': term,
        'mindate': mindate,
        'maxdate': maxdate,
        'datetype': datetype,
        'retmax': retmax,
        'usehistory': usehistory,
        'retmode': 'xml',
        'api_key': NCBI_API_KEY
    }
    return BASE_URL + 'esearch.fcgi?' + urlencode(params)

def construct_efetch_url(database, pmids, retmode):
    id_string = ','.join(pmids)
    params = {
        'db': database,
        'id': id_string,
        'retmode': retmode,
        'api_key': NCBI_API_KEY
    }
    return BASE_URL + 'efetch.fcgi?' + urlencode(params)

def make_http_request(url, retry_count=0):
  try:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.text
  except requests.exceptions.RequestException as e:
    logging.error(f"HTTP request failed: {str(e)}")
    if retry_count < MAX_RETRIES:
      logging.info(f". Retrying...(attempt {retry_count + 1})")
      time.sleep(RATE_LIMIT_DELAY * (retry_count + 1))
      return make_http_request(url, retry_count + 1)
    raise

def load_tracking_data():
  if os.path.exists(TRACKING_FILE):
    try:
      df = pd.read_csv(TRACKING_FILE)
      if 'download_method' not in df.columns:
        df['download_method'] = None
      if 'download_source_url' not in df.columns:
        df['download_source_url'] = None

      logging.info(f"Loaded {len(df)} existing records from tracking file")
      return df
    except Exception as e:
      logging.error(f"Failed to load tracking data: {str(e)}")

  #create new tracking dataframe
  df = pd.DataFrame(columns=['pmid', 'pmcid', 'pmcid_status','download_date', 'fulltext_xml_status', 'error_message', 'download_method', 'download_src_url'])
  return df

def save_tracking_data(df):
  try:
    df.to_csv(TRACKING_FILE, index=False)
    logging.info(f"Saved tracking data with {len(df)} records")
  except Exception as e:
    logging.error(f"Failed to save tracking data: {str(e)}")



def update_tracking_data(df, pmid, pmcid, pmcid_status, fulltext_xml_status,
                        error_message=None, download_method=None, download_source_url=None):
    new_record = {
        'pmid': pmid,
        'pmcid': pmcid,
        'pmcid_status': pmcid_status,
        'download_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'fulltext_xml_status': fulltext_xml_status,
        'error_message': error_message,
        'download_method': download_method,
        'download_source_url': download_source_url
    }

    if pmid in df['pmid'].values:
        idx = df[df['pmid'] == pmid].index[0]
        for key, value in new_record.items():
            df.at[idx, key] = value
        logging.debug(f"Updated existing record for PMID {pmid}")
    else:
        df = pd.concat([df, pd.DataFrame([new_record])], ignore_index=True)
        logging.debug(f"Added new record for PMID {pmid}")

    return df

def get_previous_month_date_range():
  today = datetime.now()
  first_day_current_month = today.replace(day=1)
  last_day_previous_month = first_day_current_month - timedelta(days=1)
  first_day_previous_month = last_day_previous_month.replace(day=1)

  date_start = first_day_previous_month.strftime('%Y/%m/%d')
  date_end   = last_day_previous_month.strftime('%Y/%m/%d')

  logging.info(f"Date range for previous month: {date_start} to {date_end}")

  return date_start, date_end

def parse_month_input(month_input, year_input=None):
  """function expects a month and outputs the start of the month and end of the month date)"""

  month_names ={
      'January': '01', 'jan': '01',
      'February': '02', 'feb': '02',
      'March': '03', 'mar': '03',
      'April': '04', 'apr': '04',
      'May': '05', 'may': '05',
      'June': '06', 'jun': '06',
      'July': '07', 'jul': '07',
      'August': '08', 'aug': '08',
      'September': '09', 'sep': '09',
      'October': '10', 'oct': '10',
      'November': '11', 'nov': '11',
      'December': '12', 'dec': '12'
  }

  if year_input is None:
    year = datetime.now().year
  else:
    year = int(year_input)

  month_lower = month_input.lower().strip()

  if month_lower.isdigit():
    month = int(month_lower)
    if month < 1 or month > 12:
      raise ValueError(f"Invalid month number: {month_input}.  Must be between 1 and 12")
  else:
    month = int(month_names.get(month_lower))
    if month is None :
      raise ValueError(f"Invalid month:  {month_input}")

  if year < 1900 or year > datetime.now().year:
    raise ValueError(f"Invalid year: {year_input}")

  first_day = datetime(year, month, 1)

  if month == 12:
    last_day = datetime(year + 1, 1, 1) - timedelta(days=1)
  else:
    last_day = datetime(year, month + 1, 1) - timedelta(days=1)

  date_start = first_day.strftime('%Y/%m/%d')
  date_end = last_day.strftime('%Y/%m/%d')

  logging.info(f"Parsed {month_input} {year} to range: {date_start} - {date_end}")

  return date_start, date_end

In [17]:
def search_and_download_articles(pathogen_key, date_start=None, date_end=None,
                                 interactive_mode=False, download_method='eutilities'):

    if date_start is None or date_end is None:
        date_start, date_end = get_previous_month_date_range()

    logging.info(f"Starting search for {PATHOGENS[pathogen_key]['name']}")
    logging.info(f"Date range: {date_start} to {date_end}")
    logging.info(f"Download method: {download_method.upper()}")

    search_query = build_pathogen_search_query(pathogen_key)
    tracking_df = load_tracking_data()

    search_url = construct_esearch_url(
        database=DATABASE,
        term=search_query,
        mindate=date_start,
        maxdate=date_end,
        datetype='pdat',
        retmax='100',
        usehistory=USE_HISTORY
    )

    try:
        logging.info("Executing PubMed search...")
        response = make_http_request(search_url)
        time.sleep(RATE_LIMIT_DELAY)

        total_count = extract_total_count(response)
        pmid_list = extract_pmid_list(response)

        logging.info(f"Found {total_count} total articles, retrieved {len(pmid_list)} PMIDs")

        if interactive_mode:
            print(f"\n{'='*60}")
            print(f"SEARCH RESULTS")
            print(f"{'='*60}")
            print(f"Total articles matching criteria: {total_count}")
            print(f"Articles to be downloaded: {len(pmid_list)}")
            print(f"Date range: {date_start} to {date_end}")
            print(f"Download method: {download_method.upper()}")
            print(f"{'='*60}\n")

        if not pmid_list:
            logging.warning("No articles found for the specified criteria")
            return tracking_df

        logging.info("STEP 2: Converting PMIDs to PMCIDs...")
        pmid_to_pmcid_map, missing_pmcid_list = batch_convert_pmids_to_pmcids(pmid_list)

        for pmid in missing_pmcid_list:
            tracking_df = track_missing_pmcid(tracking_df, pmid, "No PMC version available")

        if interactive_mode:
            conversion_rate = len(pmid_to_pmcid_map) / len(pmid_list) * 100 if pmid_list else 0
            print(f"\nConversion Results:")
            print(f"  Success: {len(pmid_to_pmcid_map)}/{len(pmid_list)} ({conversion_rate:.1f}%)")
            print(f"  Missing: {len(missing_pmcid_list)}")
            print()

        logging.info(f"STEP 3: Downloading full-text using {download_method.upper()} method...")

        success_count = 0
        failed_count = 0
        total_fetch = len(pmid_to_pmcid_map)

        for i, (pmid, pmcid) in enumerate(pmid_to_pmcid_map.items(), 1):

            # FIXED: Safer check for existing downloads
            existing_records = tracking_df[
                (tracking_df['pmid'] == pmid) &
                (tracking_df['pmcid'] == pmcid) &
                (tracking_df['fulltext_xml_status'] == 'success')
            ]

            if len(existing_records) > 0:
                logging.info(f"Skipping {pmcid} - already successfully downloaded")
                success_count += 1
                continue

            batch_date = datetime.now().strftime('%Y%m%d')
            download_success = False
            source_url = None

            # Execute download based on selected method
            try:
                if download_method == 'eutilities':
                    fulltext_xml = fetch_pmc_fulltext_xml(pmcid)
                    if fulltext_xml:
                        xml_success = save_xml_with_compression(fulltext_xml, pmcid, batch_date)
                        download_success = xml_success
                        source_url = construct_pmc_fetch_url(pmcid)

                elif download_method == 'aws':
                    download_success, source_url = download_from_aws_s3(pmcid, batch_date)

                elif download_method == 'ftp':
                    download_success, source_url = download_from_ftp_package(pmcid, batch_date)

                else:
                    logging.error(f"Unknown download method: {download_method}")
                    download_success = False

            except Exception as e:
                logging.error(f"Exception during download of {pmcid}: {str(e)}")
                download_success = False

            # Update tracking
            if download_success:
                tracking_df = update_tracking_data(
                    df=tracking_df,
                    pmid=pmid,
                    pmcid=pmcid,
                    pmcid_status='found',
                    fulltext_xml_status='success',
                    error_message=None,
                    download_method=download_method,
                    download_source_url=source_url
                )
                success_count += 1
                logging.info(f"Successfully processed {pmcid} via {download_method.upper()}")
            else:
                tracking_df = update_tracking_data(
                    df=tracking_df,
                    pmid=pmid,
                    pmcid=pmcid,
                    pmcid_status='found',
                    fulltext_xml_status='failed',
                    error_message=f'{download_method} download failed',
                    download_method=download_method,
                    download_source_url=source_url
                )
                failed_count += 1
                logging.error(f"Failed to process {pmcid} via {download_method.upper()}")

            # Rate limiting
            if download_method == 'eutilities':
                time.sleep(RATE_LIMIT_DELAY)
            else:
                time.sleep(FTP_RATE_LIMIT_DELAY)

            # Save progress periodically
            if i % 10 == 0:
                save_tracking_data(tracking_df)
                logging.info(f"Progress saved: {i}/{total_fetch} processed")

        save_tracking_data(tracking_df)

        logging.info(f"Processing complete:")
        logging.info(f"  Method: {download_method.upper()}")
        logging.info(f"  Full-text XML success: {success_count}")
        logging.info(f"  Full-text XML failed: {failed_count}")
        logging.info(f"  Missing PMCIDs: {len(missing_pmcid_list)}")

        generate_download_summary(tracking_df, date_start, date_end)
        return tracking_df

    except Exception as e:
        logging.error(f"Search and download failed: {str(e)}")
        import traceback
        logging.error(f"Traceback: {traceback.format_exc()}")
        save_tracking_data(tracking_df)
        raise

In [18]:
def select_download_method():
    print("\n=== Select Download Method ===")
    print("1. E-utilities (PMC efetch API) - Standard method, all PMC articles")
    print("2. AWS S3 (HTTPS) - Fast download, Open Access only")
    print("3. FTP Package - Complete packages with PDF, Open Access only")
    print("")

    choice = input("Enter your choice (1, 2, or 3): ").strip()

    if choice == '1':
      method = 'eutilities'
    elif choice == '2':
      method = 'aws'
    elif choice == '3':
      method = 'ftp'
    else:
      print("Invalid choice. Defaulting to E-utilities method.")
      method = 'eutilities'

    logging.info(f"Selected download method: {method}")
    return method

def run_interactive_mode():

    logging.info("Entering interactive mode v3a")

    print("\n=== PubMed Article Downloader v3a - Interactive Mode ===\n")

    # Step 1: Select download method
    download_method = select_download_method()
    print(f"\nSelected method: {download_method.upper()}\n")

    # Step 2: Select month and year
    month_input = input("Enter month (e.g., 'June', 'Jun', or '6'): ")
    year_input = input("Enter year (press Enter for current year): ").strip()

    if not year_input:
        year_input = None

    try:
        date_start, date_end = parse_month_input(month_input, year_input)
    except ValueError as e:
        logging.error(f"Invalid input: {e}")
        print(f"Error: {e}")
        return

    print(f"\nSearching for publications from {date_start} to {date_end}")
    print(f"Using download method: {download_method.upper()}\n")

    # Step 3: Confirm
    confirm = input("Proceed with download? (y/n): ").strip().lower()
    if confirm != 'y':
        print("Download canceled.")
        return

    # Step 4: Mount drive
    if not mount_google_drive():
        raise Exception("Failed to mount Google Drive")

    # Step 5: Process pathogen
    pathogen_combinations = ["hepatitis_a"]

    for pathogen in pathogen_combinations:
        logging.info(f"Processing pathogen: {pathogen}")

        try:
            # CRITICAL FIX: Pass download_method parameter
            tracking_df = search_and_download_articles(
                pathogen_key=pathogen,
                date_start=date_start,
                date_end=date_end,
                interactive_mode=True,
                download_method=download_method  # ← This was missing or incorrect
            )
            time.sleep(5)

        except Exception as e:
            logging.error(f"Failed to process {pathogen}: {str(e)}")
            continue

    # Step 6: Show completion message
    print("\n✓ Interactive download complete")
    print(f"Check {LOG_PATH} for detailed logs and summaries")

    if download_method == 'aws':
        print(f"Files saved to: {AWS_XML_PATH}")
    elif download_method == 'ftp':
        print(f"Files saved to: {FTP_XML_PATH}")
    else:
        print(f"Files saved to: {XML_PATH}")

def select_execution_mode():
  import sys

  if len(sys.argv) > 1:
    mode = sys.argv[1].lower()
    if mode == 'interactive':
      return 'interactive'
    elif mode == ['--scheduled', '-s']:
      return 'scheduled'

  print("\n=== PubMed Article Downloader ===")
  print("1. Interactive Mode (choose specific month)")
  print("2. Scheduled Mode (download previous month)")

  choice = input("\nSelect mode (1 or 2): ").strip()

  if choice == '1':
      return 'interactive'
  elif choice == '2':
      return 'scheduled'
  else:
      print("Invalid choice. Defaulting to scheduled mode.")
      return 'scheduled'


In [23]:
def main():

  #initialize logging
  setup_logging()
  logging.info("Starting main execution")
  logging.info("v3 Changes: PMID-to-PMCID conversion, full-text XML from PMC, removed PDF scraping")

  try:

    mode = select_execution_mode()

    if mode == 'interactive':
      run_interactive_mode()

    else:
      #mount gdrive
      if not mount_google_drive():
        raise Exception("Failed to mount Google drive")

      #run for both pathogen combinations

      pathogen_search = ["hepatitis_a"]

      for pathogen in pathogen_search:
        logging.info(f"Processing pathogen: {pathogen}")

        try:
          tracking_df = search_and_download_articles(pathogen)

          #add delay
          time.sleep(3)

        except Exception as e:
          logging.error(f"Failed to process {pathogen}: {str(e)}")
          continue

    logging.info("All processing complete")
  except Exception as e:
    logging.error(f"Main execution failed: {str(e)}")
    raise

if __name__ == "__main__":
  main()



2025-10-14 10:30:37,682 - INFO - setup_logging - Logging initialized. Log file: /content/drive/MyDrive/AI6129/logs/pubmed_download_20251014_103037.log
2025-10-14 10:30:37,687 - INFO - setup_logging - PubMed Article Downloader v3
2025-10-14 10:30:37,689 - INFO - main - Starting main execution
2025-10-14 10:30:37,691 - INFO - main - v3 Changes: PMID-to-PMCID conversion, full-text XML from PMC, removed PDF scraping



=== PubMed Article Downloader ===
1. Interactive Mode (choose specific month)
2. Scheduled Mode (download previous month)

Select mode (1 or 2): 1


2025-10-14 10:30:40,505 - INFO - run_interactive_mode - Entering interactive mode v3a



=== PubMed Article Downloader v3a - Interactive Mode ===


=== Select Download Method ===
1. E-utilities (PMC efetch API) - Standard method, all PMC articles
2. AWS S3 (HTTPS) - Fast download, Open Access only
3. FTP Package - Complete packages with PDF, Open Access only

Enter your choice (1, 2, or 3): 1


2025-10-14 10:30:42,081 - INFO - select_download_method - Selected download method: eutilities


...


Conversion Results:
  Success: 72/100 (72.0%)
  Missing: 28



2025-10-14 10:32:14,774 - INFO - fetch_pmc_fulltext_xml - Successfully retrieved full-text XML for PMC12460081
2025-10-14 10:32:14,786 - INFO - save_xml_with_compression - Saved XML for PMC12460081: 0.07 MB
2025-10-14 10:32:14,789 - INFO - search_and_download_articles - Successfully processed PMC12460081 via EUTILITIES
2025-10-14 10:32:15,133 - INFO - fetch_pmc_fulltext_xml - Fetching full text XML for 12457326
2025-10-14 10:32:16,711 - INFO - fetch_pmc_fulltext_xml - Successfully retrieved full-text XML for PMC12457326
2025-10-14 10:32:16,721 - INFO - save_xml_with_compression - Saved XML for PMC12457326: 0.11 MB
2025-10-14 10:32:16,726 - INFO - search_and_download_articles - Successfully processed PMC12457326 via EUTILITIES
2025-10-14 10:32:17,068 - INFO - fetch_pmc_fulltext_xml - Fetching full text XML for 12459786
2025-10-14 10:32:18,655 - INFO - fetch_pmc_fulltext_xml - Successfully retrieved full-text XML for PMC12459786
2025-10-14 10:32:18,666 - INFO - save_xml_with_compression 


    ===== PUBMED ARTICLE DOWNLOADER v3 - SUMMARY REPORT =====
    Date Range: 2025/01/01 to 2025/01/31
    Execution Date: 2025-10-14 10:34:31

    PMID-TO-PMCID CONVERSION:
    -------------------------
    Total PMIDs Searched: 268
    PMCIDs Found: 153 (57.1%)
    PMCIDs Not Found: 115

    FULL-TEXT XML RETRIEVAL:
    ------------------------
    Successful Downloads: 99 (36.9% of found)
    Failed Downloads: 54

    DOWNLOAD METHOD BREAKDOWN:
    --------------------------
    E-utilities Method:
    Total Attempts: 99
    Successful: 99
    Failed: 0

    AWS S3 Method:
    Total Attempts: 27
    Successful: 0
    Failed: 27

    FTP Package Method:
    Total Attempts: 27
    Successful: 0
    Failed: 27

    HISTORICAL DATA:
    ----------------
    Total Records in Database: 268
    Total Unique PMCIDs: 96

    FILES GENERATED:
    ----------------
    Tracking File: /content/drive/MyDrive/AI6129/download_tracker.csv
    Missing PMCIDs Log: /content/drive/MyDrive/AI6129/missin

2025-10-14 10:34:36,173 - INFO - main - All processing complete



✓ Interactive download complete
Check /content/drive/MyDrive/AI6129/logs for detailed logs and summaries
Files saved to: /content/drive/MyDrive/AI6129/xml
